# BusBooking · Pha 2A trên Kaggle CPU — đường đọc tìm chuyến

**Mục tiêu:** đo `GET /api/trips` có bộ lọc trên dữ liệu đủ lớn, theo mục 3.1 và 6 của [kế hoạch Pha 2](../../../docs/BusBooking_KeHoach_Pha2.md). Notebook chạy Spring Boot Java 21, PostgreSQL và Locust trong **cùng một Kaggle CPU notebook**, không cần Docker.

Notebook **tự làm toàn bộ**, không cần Add Input:
1. Clone GitHub và build JAR cho từng bản trong `VARIANTS` (hiện có P1 `fd6b5f3`, A1 `6d0e024`, A2 `5966fc5` và A3 `e78bd45`).
2. Dựng PostgreSQL riêng có `pg_stat_statements` và `auto_explain`.
3. Dùng JAR P1 chạy Flyway V1–V16, rồi sinh dữ liệu giả lập khoảng 54.000 chuyến bằng SQL có tính tất định (cùng kết quả ở mọi lần chạy).
4. Với mỗi bản: kiểm tra kết quả từng bộ lọc, ghi query plan thật của Hibernate bằng `auto_explain` (`EXPLAIN ANALYZE, BUFFERS` kèm tham số).
5. Đo tải: `wait_time = 0`, 10 và 20 users, mỗi bộ lọc chạy riêng, warm-up 60 giây rồi đo 120 giây, 3 lượt, đảo thứ tự các bản giữa các lượt. Bản có phân trang (`PAGED_VARIANTS`) đo thêm trang đầu `page=0&size=20` của hai bộ lọc trên 20 chuyến.
6. Xuất `summary.csv`, `aggregate.csv`, query plan, SQL, tài nguyên, `manifest.json` và một file ZIP.

**Cách chạy:** tạo Kaggle Notebook → **File → Import Notebook** chọn file này → Settings: **Accelerator = None (CPU)**, **Internet = On** → **Run All**. Tải file ZIP ở tab **Output** khi chạy xong.

**Muốn so P1 với bản cải tiến:** thêm commit vào `VARIANTS` ở cell cấu hình, ví dụ `('A4', '<SHA đầy đủ 40 ký tự>')`; bản hỗ trợ `page`/`size` thì thêm tên vào `PAGED_VARIANTS`. Mọi bản được build và đo trong cùng phiên, cùng dữ liệu. Thời gian tăng theo số bản: với P1, A1, A2, A3, phần đo tải khoảng 4 giờ 30 phút (ô đo tải in ước tính). Có thể bỏ bớt bản để chạy nhanh hơn, nhưng mỗi phiên phải có P1. Không ghép số của hai phiên Kaggle khác nhau thành một cặp so sánh vì CPU có thể khác.

**Cảnh báo:** PostgreSQL ở đây là cluster riêng tạo trong notebook. Không trỏ notebook tới database ngoài.

## 1. Cấu hình

Chỉ sửa cell này. `QUICK_CHECK = True` chạy thử nhanh để kiểm tra pipeline (1 lượt, 10 users, đo 20 giây); số liệu của chế độ này **không dùng để báo cáo**.

In [ ]:
import base64, csv, hashlib, json, math, os, platform, re, secrets, shutil, socket, subprocess, sys, tempfile, time, zipfile
from contextlib import contextmanager
from datetime import datetime, timezone
from pathlib import Path
from statistics import median

REPO_URL = 'https://github.com/thanhphu25/Bus-booking-system.git'

# Các bản đo trong CÙNG một phiên. Bản đầu tiên phải là P1 (đối chứng gốc).
# Khi có commit cải tiến, thêm dòng mới, ví dụ: ('A4', '<SHA đầy đủ 40 ký tự>').
VARIANTS = [
    ('P1', 'fd6b5f3026412d2359d70bfad681fc0773c8bcae'),
    ('A1', '6d0e02419a488eeb90cc0c2552bb4f99dc92356b'),  # V18: index trips (departure_time, route_id)
    ('A2', '5966fc5eb237fb165b98624f73795a0206107cf5'),  # page/size tùy chọn, thứ tự departure_time, id
    ('A3', 'e78bd45619b65a4b48af0a403d444e38a6d9f155'),  # TripSearchQuery + projection thay cho entity graph
]
# Các bản hỗ trợ page/size. Với các bản này, notebook đo thêm trang đầu (page=0&size=PAGE_SIZE) của
# bộ lọc trên PAGE_SIZE chuyến và yêu cầu thứ tự departureTime rồi id.
PAGED_VARIANTS = {'A2', 'A3'}
PAGE_SIZE = 20

USER_LEVELS = [10, 20]     # plan 6.2: wait_time = 0, 10 và 20 virtual users
WARMUP_SECONDS = 60        # plan 6.2: warm-up 60 giây
MEASURE_SECONDS = 120      # plan 6.2: đo 120 giây
REPETITIONS = 3            # plan 6.2: ba lượt, đảo thứ tự giữa các lượt
TRACE_ROUNDS = 15          # số vòng gửi tuần tự mọi bộ lọc khi ghi query plan
QUICK_CHECK = False

if QUICK_CHECK:
    USER_LEVELS, WARMUP_SECONDS, MEASURE_SECONDS, REPETITIONS, TRACE_ROUNDS = [10], 10, 20, 1, 3

FILTER_DAY = '2026-12-15'  # ngày dùng cho các bộ lọc, nằm trong khoảng dữ liệu giả lập
JAVA_FLAGS = ['-Xms256m', '-Xmx1024m', '-Duser.timezone=UTC']
HIKARI_POOL = 10
API = 'http://127.0.0.1:8085'
PG_PORT = 55434
PACKAGES = ['locust==2.32.10', 'psutil==6.1.1', 'requests==2.32.3', 'psycopg[binary]==3.2.4']

if VARIANTS[0][0] != 'P1':
    raise ValueError('Bản đầu tiên trong VARIANTS phải là P1')
if len({name for name, _ in VARIANTS}) != len(VARIANTS):
    raise ValueError('Tên bản trong VARIANTS bị trùng')
if 'P1' in PAGED_VARIANTS:
    raise ValueError('P1 không có page/size, không được nằm trong PAGED_VARIANTS')
for name, sha in VARIANTS:
    if not re.fullmatch(r'[A-Za-z0-9_-]+', name) or not re.fullmatch(r'[0-9a-f]{40}', sha):
        raise ValueError(f'Tên hoặc SHA không hợp lệ: {name} {sha}. Dùng SHA đầy đủ 40 ký tự.')

WORKING = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()
RUN_ID = datetime.now(timezone.utc).strftime('phase2-a-%Y%m%dT%H%M%SZ')
OUTPUT = WORKING / 'busbooking-phase2-a' / RUN_ID
OUTPUT.mkdir(parents=True)
SCRATCH = Path(tempfile.mkdtemp(prefix='busbooking-phase2-a-'))


def run(command, **kwargs):
    return subprocess.run([str(part) for part in command], check=True, text=True, **kwargs)


def utc():
    return datetime.now(timezone.utc).isoformat()


def write_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, default=str), encoding='utf-8')


def write_csv(path, rows, columns=None):
    path.parent.mkdir(parents=True, exist_ok=True)
    columns = columns or (list(rows[0]) if rows else [])
    with path.open('w', encoding='utf-8', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=columns, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(rows)


def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()


def sha256_file(path):
    return sha256_bytes(Path(path).read_bytes())


print('Output:', OUTPUT)
print('OS:', platform.platform(), '| Python:', platform.python_version(), '| CPU logical:', os.cpu_count())
subprocess.run(['bash', '-lc', 'lscpu | grep -E "Model name|^CPU\\(s\\)|Architecture" | head -6'], check=False)

## 2. Cài công cụ: Locust, Java 21, PostgreSQL

Phiên bản thư viện Python được ghim giống bài đo phần B để số liệu của nhóm dùng cùng công cụ. Nếu máy chưa có Java 21, tải Amazon Corretto 21. PostgreSQL cần bản 15 trở lên (để ghi log dạng JSON) và có gói `contrib` (chứa `pg_stat_statements`, `auto_explain`).

Trên Kaggle, `pip` có thể in một khối chữ đỏ bắt đầu bằng `ERROR: pip's dependency resolver does not currently take into account…`. Đó là cảnh báo xung đột với các gói Kaggle cài sẵn (`google-colab`, `bigframes`…) mà notebook không dùng; cell vẫn chạy tiếp. Lỗi thật sẽ dừng cell và có `Traceback`.

In [ ]:
run([sys.executable, '-m', 'pip', 'install', '-q', *PACKAGES])
import psutil, psycopg, requests
from psycopg import sql
from psycopg.rows import dict_row

java = shutil.which('java')
if not java or not re.search(r'version "21(?:[."]|$)', run([java, '-version'], capture_output=True).stderr):
    java_dir = SCRATCH / 'jdk21'
    java = java_dir / 'bin/java'
    if not java.is_file():
        java_dir.mkdir(exist_ok=True)
        archive = SCRATCH / 'jdk21.tar.gz'
        run(['curl', '-fL', '--retry', '3', '-o', archive,
             'https://corretto.aws/downloads/latest/amazon-corretto-21-x64-linux-jdk.tar.gz'])
        run(['tar', '-xzf', archive, '--strip-components=1', '-C', java_dir])
        archive.unlink()
java = Path(java).resolve()
JAVA_VERSION = run([java, '-version'], capture_output=True).stderr.strip()
print(JAVA_VERSION)


def postgres_bins():
    found = []
    for initdb in Path('/usr/lib/postgresql').glob('[0-9]*/bin/initdb'):
        version = initdb.parent.parent.name
        lib = initdb.parent.parent / 'lib'
        if (int(version) >= 15 and (lib / 'auto_explain.so').is_file()
                and Path('/usr/share/postgresql', version, 'extension', 'pg_stat_statements.control').is_file()):
            found.append(initdb.parent)
    return found


if not postgres_bins():
    run(['apt-get', 'update', '-qq'])
    run(['apt-get', 'install', '-y', '-qq', 'postgresql', 'postgresql-contrib'],
        env={**os.environ, 'DEBIAN_FRONTEND': 'noninteractive'})
if not postgres_bins():
    raise RuntimeError('Không tìm thấy PostgreSQL >= 15 có auto_explain và pg_stat_statements')
PG_BIN = max(postgres_bins(), key=lambda path: int(path.parent.name))
print('PostgreSQL executables:', PG_BIN)

## 3. Clone GitHub và build JAR cho từng bản

Mỗi bản được checkout **sạch** đúng SHA đã ghim rồi build bằng Maven Wrapper (`-DskipTests`). Notebook từ chối chạy nếu commit không khớp hoặc working tree không sạch. SHA-256 của JAR được ghi vào manifest.

In [ ]:
repository = SCRATCH / 'repo'
git_env = {**os.environ, 'GIT_TERMINAL_PROMPT': '0'}
run(['git', 'clone', '--no-checkout', '--', REPO_URL, repository], env=git_env)
build_env = {**os.environ, 'JAVA_HOME': str(java.parent.parent),
             'PATH': str(java.parent) + os.pathsep + os.environ.get('PATH', '')}

JARS = {}
SOURCES = {}
for name, sha in VARIANTS:
    checkout = SCRATCH / f'src-{name}'
    run(['git', '-C', repository, 'worktree', 'add', '--detach', checkout, sha], env=git_env)
    actual = run(['git', 'rev-parse', 'HEAD'], cwd=checkout, capture_output=True).stdout.strip()
    dirty = run(['git', 'status', '--porcelain'], cwd=checkout, capture_output=True).stdout
    if actual != sha or dirty:
        raise RuntimeError(f'Checkout của {name} phải sạch và khớp {sha}')
    print(f'Đang build {name} từ {sha} (mất vài phút)...', flush=True)
    run(['sh', './mvnw', '-B', '-q', '-DskipTests', 'clean', 'package'], cwd=checkout, env=build_env)
    jar = checkout / 'target' / 'busbooking-0.0.1-SNAPSHOT.jar'
    with zipfile.ZipFile(jar) as archive:
        if 'BOOT-INF/classes/application.properties' not in archive.namelist():
            raise RuntimeError(f'JAR Spring Boot không hợp lệ: {jar}')
    JARS[name] = jar.rename(jar.with_name(f'busbooking-{name}.jar'))
    SOURCES[name] = {'source_sha': sha, 'working_tree': 'clean', 'jar_name': JARS[name].name,
                     'jar_sha256': sha256_file(JARS[name])}
    print(name, SOURCES[name])

## 4. Khởi động PostgreSQL riêng

Tạo cluster mới bằng `initdb` (cổng 55434, chỉ nghe `127.0.0.1`), nạp sẵn `pg_stat_statements` và `auto_explain`. `auto_explain` **tắt theo mặc định**; notebook chỉ bật nó trên database dùng để ghi query plan, không bật trong lúc đo tải. Log server ghi dạng JSON để đọc lại plan. Mật khẩu ngẫu nhiên chỉ nằm trong RAM, không ghi vào kết quả.

In [ ]:
import pwd

pg_user = pwd.getpwnam('postgres')
PG_ROOT = Path(tempfile.mkdtemp(prefix='busbooking-phase2-a-pg-'))
os.chown(PG_ROOT, pg_user.pw_uid, pg_user.pw_gid)
PG_DATA = PG_ROOT / 'data'
PG_LOG_DIR = PG_ROOT / 'log'
PG_LOG_DIR.mkdir()
os.chown(PG_LOG_DIR, pg_user.pw_uid, pg_user.pw_gid)
AS_POSTGRES = ['runuser', '-u', 'postgres', '--']
PG_PASSWORD = secrets.token_urlsafe(30)
password_file = PG_ROOT / 'password'
password_file.write_text(PG_PASSWORD + '\n', encoding='ascii')
password_file.chmod(0o600)
os.chown(password_file, pg_user.pw_uid, pg_user.pw_gid)
try:
    run([*AS_POSTGRES, PG_BIN / 'initdb', '-D', PG_DATA, '-U', 'postgres', '--encoding=UTF8',
         '--locale=C.UTF-8', '--auth-local=peer', '--auth-host=scram-sha-256', f'--pwfile={password_file}'])
finally:
    password_file.unlink(missing_ok=True)

server_options = ' '.join([
    f'-p {PG_PORT}', '-h 127.0.0.1',
    '-c shared_preload_libraries=pg_stat_statements,auto_explain',
    '-c logging_collector=on', '-c log_destination=jsonlog',
    f'-c log_directory={PG_LOG_DIR}', '-c log_filename=postgresql.log', '-c log_timezone=UTC',
])
run([*AS_POSTGRES, PG_BIN / 'pg_ctl', '-D', PG_DATA, '-l', PG_ROOT / 'startup.log', '-w', 'start', '-o', server_options])
PG_PID = int((PG_DATA / 'postmaster.pid').read_text().splitlines()[0])


def connect(db='postgres'):
    return psycopg.connect(host='127.0.0.1', port=PG_PORT, dbname=db, user='postgres', password=PG_PASSWORD,
                           autocommit=True, row_factory=dict_row)


with connect() as admin:
    PG_SETTINGS = admin.execute('''
        SELECT name, setting, unit FROM pg_settings WHERE name IN (
            'server_version', 'default_transaction_isolation', 'shared_buffers', 'work_mem',
            'effective_cache_size', 'random_page_cost', 'plan_cache_mode', 'jit', 'max_connections',
            'shared_preload_libraries', 'pg_stat_statements.track', 'auto_explain.log_min_duration')
        ORDER BY name''').fetchall()
for row in PG_SETTINGS:
    print(row['name'], '=', row['setting'], row['unit'] or '')

## 5. Hàm dùng chung: khởi động backend, Locust, đo tài nguyên và SQL

- **Backend:** mỗi lần chạy một JAR với cùng cờ JVM, pool Hikari 10 và profile `dev`. Notebook kiểm tra cổng 8085 còn trống trước khi khởi động để không đo nhầm một backend cũ còn sót.
- **Locust:** `FastHttpUser`, `wait_time = 0`. Mỗi request được ghi lại thời gian để tính p50/p95/p99 chính xác (không dùng bảng phần trăm làm tròn của Locust).
- **Tài nguyên:** lấy mẫu CPU giây và RSS của Java, PostgreSQL (tiến trình chính và các tiến trình con) và Locust mỗi 0,5 giây trong cửa sổ đo.
- **SQL:** chênh lệch `pg_stat_statements` của user ứng dụng trong database đang đo, trước và sau mỗi cửa sổ đo.

In [ ]:
LOCUSTFILE = OUTPUT / 'locustfile.py'
LOCUSTFILE.write_text('''import csv, json, os, random
from locust import FastHttpUser, constant, events, task

PATHS = json.loads(os.environ['A_PATHS'])
RECORDS = []


@events.request.add_listener
def record(request_type, name, response_time, response_length, exception, **kwargs):
    RECORDS.append((name, round(response_time, 3), response_length, '' if exception is None else str(exception)[:200]))


@events.quitting.add_listener
def dump(environment, **kwargs):
    with open(os.environ['A_REQUESTS_CSV'], 'w', newline='', encoding='utf-8') as stream:
        writer = csv.writer(stream)
        writer.writerow(['name', 'response_ms', 'response_bytes', 'error'])
        writer.writerows(RECORDS)


class SearchUser(FastHttpUser):
    wait_time = constant(0)

    @task
    def search(self):
        name, path = random.choice(PATHS)
        with self.client.get(path, name=name, catch_response=True) as response:
            if response.status_code != 200:
                response.failure(f'HTTP {response.status_code}')
''', encoding='utf-8')

APP_DB_USER = 'busb_a_app'
APP_DB_PASSWORD = secrets.token_urlsafe(30)
APP_ENV = {**os.environ, 'SPRING_PROFILES_ACTIVE': 'dev', 'SERVER_PORT': '8085', 'SERVER_ADDRESS': '127.0.0.1',
           'SPRING_DATASOURCE_HIKARI_MAXIMUM_POOL_SIZE': str(HIKARI_POOL),
           'JWT_SECRET': base64.b64encode(secrets.token_bytes(32)).decode('ascii'),
           'EMAIL_OTP_SECRET': base64.b64encode(secrets.token_bytes(32)).decode('ascii'),
           'BOOTSTRAP_SUPER_ADMIN_ENABLED': 'false', 'API_DOCS_ENABLED': 'false',
           'DB_USERNAME': APP_DB_USER, 'DB_PASSWORD': APP_DB_PASSWORD}
READY_PATH = '/api/trips?departureDate=2000-01-01'


def port_in_use(port):
    with socket.socket() as probe:
        probe.settimeout(1)
        return probe.connect_ex(('127.0.0.1', port)) == 0


@contextmanager
def backend(variant, db, log_path):
    if port_in_use(8085):
        raise RuntimeError('Cổng 8085 đang bận: còn backend cũ chạy. Restart kernel rồi chạy lại từ đầu.')
    env = {**APP_ENV, 'DB_URL': f'jdbc:postgresql://127.0.0.1:{PG_PORT}/{db}'}
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open('w', encoding='utf-8') as stream:
        process = subprocess.Popen([str(java), *JAVA_FLAGS, '-jar', str(JARS[variant])], env=env,
                                   stdout=stream, stderr=subprocess.STDOUT)
    try:
        deadline = time.monotonic() + 180
        while True:
            if process.poll() is not None:
                raise RuntimeError(f'Backend {variant} dừng khi khởi động, xem {log_path}:\n' +
                                   '\n'.join(log_path.read_text(errors='replace').splitlines()[-40:]))
            if time.monotonic() > deadline:
                raise RuntimeError(f'Backend {variant} không phản hồi sau 180 giây, xem {log_path}')
            try:
                if requests.get(API + READY_PATH, timeout=2).status_code == 200:
                    break
            except requests.RequestException:
                pass
            time.sleep(0.5)
        yield process
    finally:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=30)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()


def sample_resources(groups):
    timestamp = time.monotonic_ns()
    rows = []
    for group, pid in groups.items():
        try:
            root = psutil.Process(pid)
            members = [root, *root.children(recursive=True)] if group == 'postgresql' else [root]
        except psutil.NoSuchProcess:
            continue
        for process in members:
            try:
                with process.oneshot():
                    cpu = process.cpu_times()
                    rows.append({'timestamp_ns': timestamp, 'group': group, 'pid': process.pid,
                                 'cpu_seconds': cpu.user + cpu.system, 'rss_bytes': process.memory_info().rss})
            except psutil.NoSuchProcess:
                continue
    return rows


def resource_summary(samples):
    frames = {}
    for row in samples:
        frames.setdefault(row['group'], {}).setdefault(row['timestamp_ns'], {})[row['pid']] = row
    summary = {}
    for group, by_time in frames.items():
        ordered = [by_time[key] for key in sorted(by_time)]
        seconds = (max(by_time) - min(by_time)) / 1e9 if len(by_time) > 1 else 0
        cpu = sum(max(0, after[pid]['cpu_seconds'] - before[pid]['cpu_seconds'])
                  for before, after in zip(ordered, ordered[1:]) for pid in before.keys() & after.keys())
        summary[group] = {'cpu_seconds': cpu, 'cpu_cores_avg': cpu / seconds if seconds else None,
                          'peak_rss_mib': max(sum(r['rss_bytes'] for r in frame.values()) for frame in ordered) / 1024**2}
    return summary


def percentile(values, percent):
    if not values:
        return None
    ordered = sorted(values)
    return ordered[max(0, math.ceil(percent * len(ordered) / 100) - 1)]


def run_locust(paths, users, seconds, prefix, java_pid, measure):
    env = {**os.environ, 'A_PATHS': json.dumps(paths), 'A_REQUESTS_CSV': str(prefix) + '_requests.csv',
           'PYTHONDONTWRITEBYTECODE': '1'}
    command = [sys.executable, '-m', 'locust', '-f', LOCUSTFILE, '--headless', '--only-summary', '--host', API,
               '--users', users, '--spawn-rate', users, '--run-time', f'{seconds}s', '--stop-timeout', '10',
               '--csv', prefix, '--exit-code-on-error', '0']
    samples = []
    with open(str(prefix) + '_locust.log', 'w', encoding='utf-8') as log:
        process = subprocess.Popen([str(part) for part in command], env=env, stdout=log, stderr=subprocess.STDOUT)
        started = time.monotonic()
        while process.poll() is None:
            if measure:
                samples.extend(sample_resources({'java': java_pid, 'postgresql': PG_PID, 'locust': process.pid}))
            time.sleep(0.5)
        elapsed = time.monotonic() - started
    if process.returncode != 0:
        raise RuntimeError(f'Locust lỗi, xem {prefix}_locust.log')
    with open(str(prefix) + '_requests.csv', encoding='utf-8', newline='') as stream:
        records = list(csv.DictReader(stream))
    ok = [float(row['response_ms']) for row in records if not row['error']]
    result = {'requests': len(records), 'failures': len(records) - len(ok),
              'rps': len(records) / seconds, 'successful_rps': len(ok) / seconds,
              'p50_ms': percentile(ok, 50), 'p95_ms': percentile(ok, 95), 'p99_ms': percentile(ok, 99),
              'avg_response_bytes': (sum(int(row['response_bytes']) for row in records if not row['error']) / len(ok)) if ok else None,
              'locust_wall_seconds': elapsed}
    if measure:
        write_csv(Path(str(prefix) + '_resources.csv'), samples, ['timestamp_ns', 'group', 'pid', 'cpu_seconds', 'rss_bytes'])
        for group, values in resource_summary(samples).items():
            for key, value in values.items():
                result[f'{group}_{key}'] = value
    return result


def statements(connection):
    rows = connection.execute('''
        SELECT queryid, query, calls, total_exec_time AS exec_ms, rows, shared_blks_hit, shared_blks_read
        FROM pg_stat_statements
        WHERE dbid = (SELECT oid FROM pg_database WHERE datname = current_database())
          AND userid = (SELECT oid FROM pg_roles WHERE rolname = %s)''', (APP_DB_USER,)).fetchall()
    return {row['queryid']: row for row in rows}


def statement_delta(before, after):
    keys = ('calls', 'exec_ms', 'rows', 'shared_blks_hit', 'shared_blks_read')
    changed = []
    for query_id, row in after.items():
        previous = before.get(query_id, {})
        delta = {**row, **{key: row[key] - previous.get(key, 0) for key in keys}}
        if delta['calls'] > 0:
            changed.append(delta)
    return sorted(changed, key=lambda row: (-row['calls'], row['query']))


def flyway_versions(connection):
    rows = connection.execute('SELECT version FROM flyway_schema_history WHERE success AND version IS NOT NULL '
                              'ORDER BY installed_rank').fetchall()
    return [row['version'] for row in rows]


print('Locustfile SHA-256:', sha256_file(LOCUSTFILE))

## 6. Dựng snapshot P1 và sinh dữ liệu giả lập

**Snapshot:** chạy JAR P1 trên database rỗng để Flyway tạo schema V1–V16, dừng backend, rồi nạp dữ liệu. Mỗi lượt đo sau đó tạo database mới bằng `CREATE DATABASE … TEMPLATE` từ snapshot này, nên mọi bản bắt đầu từ cùng một trạng thái và tự chạy migration của mình.

**Dữ liệu (theo plan 3.1):** tính tất định bằng `md5`, không dùng `random()`, nên lần chạy nào cũng ra đúng cùng dữ liệu.

| Bảng | Số dòng | Cách sinh |
|---|---:|---|
| `locations` | 40 | `Station 01…40` |
| `bus_operators` | 10 | |
| `vehicles` | 200 | 3 loại xe, 9/40/45 ghế |
| `routes` | 120 | Mỗi điểm đi có 3 điểm đến; 3 tuyến đầu là **tuyến đông** |
| `trips` | khoảng 54.000 | 182 ngày từ 2026-10-01; tuyến đông 36–44 chuyến/ngày, tuyến thường 0–3 chuyến/ngày; giờ chạy theo bước 5 phút nên có chuyến **trùng giờ** |

Trạng thái chuyến: trước 2026-11-15 là `COMPLETED`, khoảng 4% `CANCELLED`, còn lại `SCHEDULED`. Truy vấn tìm chuyến hiện không lọc theo trạng thái nên vẫn trả cả các chuyến này.

**Bộ lọc** được chọn tự động trên ngày `FILTER_DAY`: không có kết quả, 1–20 kết quả, trên 20 kết quả (tuyến đông), chỉ lọc ngày, và không lọc. Bộ lọc **không lọc** trả toàn bộ khoảng 54.000 chuyến nên chỉ dùng để kiểm tra tương thích và ghi query plan, không đưa vào đo tải.

**Trang đầu** (`*_page0`): cùng bộ lọc thêm `page=0&size=PAGE_SIZE`, chỉ gửi tới bản trong `PAGED_VARIANTS`. Đo tải trang đầu của tuyến đông và chỉ lọc ngày (hai bộ lọc trên 20 chuyến) để so với chính bộ lọc đó khi không phân trang trong cùng bản, theo cặp A2 của plan. 20 ID đúng của mỗi trang được tính trước bằng SQL `ORDER BY departure_time, id LIMIT 20` trên snapshot.

In [ ]:
FIXTURE_SQL = '''
SET TIME ZONE 'UTC';

INSERT INTO locations (name, province, address)
SELECT 'Station ' || lpad(n::text, 2, '0'), 'Province ' || lpad(n::text, 2, '0'), 'Address ' || lpad(n::text, 2, '0')
FROM generate_series(1, 40) AS n
ORDER BY n;

INSERT INTO bus_operators (name, address, phone, email, status, created_at)
SELECT 'Operator ' || lpad(n::text, 2, '0'), 'Operator address ' || lpad(n::text, 2, '0'),
       '09' || lpad(n::text, 8, '0'), 'operator' || lpad(n::text, 2, '0') || '@example.test',
       'ACTIVE', TIMESTAMP '2026-09-01 00:00:00'
FROM generate_series(1, 10) AS n
ORDER BY n;

INSERT INTO vehicles (operator_id, license_plate, vehicle_type, seat_capacity, status, created_at)
SELECT o.id, 'BENCHA' || lpad(n::text, 4, '0'),
       (ARRAY['SLEEPER', 'SEATER', 'LIMOUSINE'])[1 + mod(n, 3)],
       (ARRAY[40, 45, 9])[1 + mod(n, 3)],
       'ACTIVE', TIMESTAMP '2026-09-01 00:00:00'
FROM generate_series(1, 200) AS n
JOIN (SELECT id, row_number() OVER (ORDER BY id) AS rn FROM bus_operators) AS o ON o.rn = 1 + mod(n - 1, 10)
ORDER BY n;

INSERT INTO routes (origin_id, destination_id, estimated_duration_minutes, status)
SELECT o.id, d.id, 60 + mod(i * 13 + k * 29, 600), 'ACTIVE'
FROM generate_series(1, 40) AS i
CROSS JOIN generate_series(1, 3) AS k
JOIN (SELECT id, row_number() OVER (ORDER BY id) AS rn FROM locations) AS o ON o.rn = i
JOIN (SELECT id, row_number() OVER (ORDER BY id) AS rn FROM locations) AS d ON d.rn = 1 + mod(i - 1 + 7 * k, 40)
ORDER BY k, i;

INSERT INTO trips (route_id, vehicle_id, departure_time, arrival_time, price, available_seats, status, created_at)
WITH r AS (
    SELECT id, row_number() OVER (ORDER BY id) AS rn, estimated_duration_minutes AS duration FROM routes
), v AS (
    SELECT id, row_number() OVER (ORDER BY id) - 1 AS vn, seat_capacity FROM vehicles
), route_day AS (
    SELECT r.id AS route_id, r.rn, r.duration, d,
           ('x' || substr(md5(r.rn::text || ':' || d::text), 1, 7))::bit(28)::int AS h
    FROM r CROSS JOIN generate_series(0, 181) AS d
), slot AS (
    SELECT rd.*, k
    FROM route_day AS rd
    CROSS JOIN LATERAL generate_series(1, CASE WHEN rd.rn <= 3 THEN 36 + mod(rd.h, 9) ELSE mod(rd.h, 4) END) AS k
), timed AS (
    SELECT s.*, (DATE '2026-10-01' + s.d) + TIME '05:00' + make_interval(mins => 5 * mod(s.h + 37 * s.k, 216)) AS departure
    FROM slot AS s
)
SELECT t.route_id, v.id, t.departure, t.departure + make_interval(mins => t.duration),
       100000 + 10000 * mod(t.h + t.k, 30),
       mod(t.h + 7 * t.k, v.seat_capacity + 1),
       CASE WHEN mod(t.h + t.k, 25) = 0 THEN 'CANCELLED'
            WHEN t.departure < TIMESTAMP '2026-11-15 00:00:00' THEN 'COMPLETED'
            ELSE 'SCHEDULED' END,
       TIMESTAMP '2026-09-01 00:00:00'
FROM timed AS t
JOIN v ON v.vn = mod(t.rn * 31 + t.d * 7 + t.k, 200)
ORDER BY t.d, t.rn, t.k;
'''
(OUTPUT / 'fixture.sql').write_text(FIXTURE_SQL, encoding='utf-8')

BASE_DB = 'busb_a_base'
with connect() as admin:
    admin.execute(sql.SQL('CREATE ROLE {} LOGIN PASSWORD {}').format(sql.Identifier(APP_DB_USER), sql.Literal(APP_DB_PASSWORD)))
    admin.execute(sql.SQL('CREATE DATABASE {} OWNER {}').format(sql.Identifier(BASE_DB), sql.Identifier(APP_DB_USER)))

print('Chạy Flyway V1–V16 bằng JAR P1...', flush=True)
with backend('P1', BASE_DB, OUTPUT / 'snapshot_backend.log'):
    pass

with connect(BASE_DB) as connection:
    versions = flyway_versions(connection)
    if versions[-1:] != ['16']:
        raise RuntimeError(f'JAR P1 phải tạo đúng schema V1–V16, nhận được {versions}')
    if connection.execute('SELECT count(*) AS n FROM trips').fetchone()['n']:
        raise RuntimeError('Snapshot phải bắt đầu từ bảng trips rỗng')
    connection.execute('CREATE EXTENSION pg_stat_statements')
    started = time.monotonic()
    connection.execute(FIXTURE_SQL)
    connection.execute('VACUUM ANALYZE')
    print(f'Nạp dữ liệu và VACUUM ANALYZE: {time.monotonic() - started:.1f} giây')

    counts = {table: connection.execute(sql.SQL('SELECT count(*) AS n FROM {}').format(sql.Identifier(table))).fetchone()['n']
              for table in ('locations', 'bus_operators', 'vehicles', 'routes', 'trips', 'bookings')}
    per_route = connection.execute('''
        SELECT r.id, r.origin_id, r.destination_id,
               count(t.id) FILTER (WHERE t.departure_time >= %(day)s::date AND t.departure_time < %(day)s::date + 1) AS n
        FROM routes AS r LEFT JOIN trips AS t ON t.route_id = r.id
        GROUP BY r.id ORDER BY r.id''', {'day': FILTER_DAY}).fetchall()
    day_total = connection.execute('''
        SELECT count(*) AS n FROM trips
        WHERE departure_time >= %(day)s::date AND departure_time < %(day)s::date + 1''', {'day': FILTER_DAY}).fetchone()['n']
    ties = connection.execute('''
        SELECT count(*) AS n FROM (SELECT departure_time FROM trips GROUP BY departure_time HAVING count(*) > 1) AS x''').fetchone()['n']
    span = connection.execute('SELECT min(departure_time) AS first, max(departure_time) AS last FROM trips').fetchone()
    statuses = {row['status']: row['n'] for row in connection.execute(
        'SELECT status, count(*) AS n FROM trips GROUP BY status ORDER BY status').fetchall()}
    table_bytes = {row['relname']: row['bytes'] for row in connection.execute('''
        SELECT relname, pg_total_relation_size(oid) AS bytes FROM pg_class
        WHERE relname IN ('trips', 'routes', 'locations', 'vehicles', 'bus_operators')''').fetchall()}

if counts['bookings'] or not 45000 <= counts['trips'] <= 65000:
    raise RuntimeError(f'Dữ liệu giả lập không như thiết kế: {counts}')


def pick(predicate, label):
    for row in per_route:
        if predicate(row['n']):
            return row
    raise RuntimeError(f'Không tìm được tuyến cho bộ lọc {label} trên {FILTER_DAY}')


zero, small, large = pick(lambda n: n == 0, 'zero'), pick(lambda n: 2 <= n <= 20, '1-20'), pick(lambda n: n > 20, '>20')
FILTERS = {
    'zero': ({'originId': zero['origin_id'], 'destinationId': zero['destination_id'], 'departureDate': FILTER_DAY}, 0),
    'small_1_20': ({'originId': small['origin_id'], 'destinationId': small['destination_id'], 'departureDate': FILTER_DAY}, small['n']),
    'large_gt_20': ({'originId': large['origin_id'], 'destinationId': large['destination_id'], 'departureDate': FILTER_DAY}, large['n']),
    'date_only': ({'departureDate': FILTER_DAY}, day_total),
    'no_filter': ({}, counts['trips']),
}
FILTER_ROUTE = {'zero': zero['id'], 'small_1_20': small['id'], 'large_gt_20': large['id']}
LOAD_FILTERS = ['zero', 'small_1_20', 'large_gt_20', 'date_only']

# Trang đầu của các bộ lọc trên PAGE_SIZE chuyến; chỉ gửi tới bản trong PAGED_VARIANTS.
PAGE_OF = {}
for name in ('large_gt_20', 'date_only', 'no_filter'):
    params, expected = FILTERS[name]
    FILTERS[f'{name}_page0'] = ({**params, 'page': 0, 'size': PAGE_SIZE}, min(expected, PAGE_SIZE))
    PAGE_OF[f'{name}_page0'] = name
    if name in FILTER_ROUTE:
        FILTER_ROUTE[f'{name}_page0'] = FILTER_ROUTE[name]
PAGED_LOAD_FILTERS = ['large_gt_20_page0', 'date_only_page0']

FIRST_PAGE_IDS = {}
with connect(BASE_DB) as connection:
    for page_name, name in PAGE_OF.items():
        params = FILTERS[name][0]
        conditions, values = ['TRUE'], {'size': PAGE_SIZE}
        if 'originId' in params:
            conditions.append('r.origin_id = %(origin)s AND r.destination_id = %(destination)s')
            values.update(origin=params['originId'], destination=params['destinationId'])
        if 'departureDate' in params:
            conditions.append('t.departure_time >= %(day)s::date AND t.departure_time < %(day)s::date + 1')
            values['day'] = params['departureDate']
        query = ('SELECT t.id FROM trips AS t JOIN routes AS r ON r.id = t.route_id WHERE '
                 + ' AND '.join(conditions) + ' ORDER BY t.departure_time, t.id LIMIT %(size)s')
        FIRST_PAGE_IDS[page_name] = [row['id'] for row in connection.execute(query, values).fetchall()]


def load_filters(variant):
    return LOAD_FILTERS + (PAGED_LOAD_FILTERS if variant in PAGED_VARIANTS else [])


def filter_path(name):
    params = FILTERS[name][0]
    return '/api/trips' + ('?' + '&'.join(f'{key}={value}' for key, value in params.items()) if params else '')


FIXTURE_STATS = {'counts': counts, 'departure_span': span, 'status_counts': statuses,
                 'departure_times_shared_by_several_trips': ties, 'total_relation_bytes': table_bytes,
                 'filter_day': FILTER_DAY, 'fixture_sha256': sha256_bytes(FIXTURE_SQL.encode('utf-8')),
                 'page_size': PAGE_SIZE,
                 'filters': {name: {'path': filter_path(name), 'expected_rows': expected, 'route_id': FILTER_ROUTE.get(name),
                                    'load_tested': name in LOAD_FILTERS or name in PAGED_LOAD_FILTERS,
                                    'paged_variants_only': name in PAGE_OF,
                                    'first_page_ids': FIRST_PAGE_IDS.get(name)}
                             for name, (_, expected) in FILTERS.items()}}
write_json(OUTPUT / 'fixture_stats.json', FIXTURE_STATS)
print(json.dumps(FIXTURE_STATS, ensure_ascii=False, indent=2, default=str))

## 7. Kiểm tra kết quả và ghi query plan thật của từng bản

Với mỗi bản, notebook tạo một database riêng từ snapshot, **bật `auto_explain` chỉ trên database này** (`log_analyze`, `log_buffers`, định dạng JSON) rồi khởi động backend:

1. **Kiểm tra kết quả:** gọi từng bộ lọc đo tải một lần; số chuyến phải bằng số đếm bằng SQL, mọi chuyến phải đúng ngày và đúng tuyến của bộ lọc. Thứ tự: P1 và A1 chỉ cần `departureTime` không giảm; bản trong `PAGED_VARIANTS` phải tăng chặt theo `departureTime` rồi `id`, và mỗi trang đầu phải đúng 20 ID tính bằng SQL. Danh sách ID được lưu để so giữa các bản: không phân trang thì bản cải tiến phải trả đúng tập chuyến như P1, và **nội dung JSON từng chuyến** phải giống P1 (so bằng SHA-256 của JSON đã sắp theo `id`). Trang đầu được so với 20 chuyến đầu khi sắp kết quả đầy đủ của P1 theo `departureTime` rồi `id`, như cặp P1/A3 của plan.
2. **Query plan:** gửi tuần tự các bộ lọc đo tải `TRACE_ROUNDS` vòng, mỗi request được gắn với plan PostgreSQL ghi trong khoảng thời gian của nó. Plan này là SQL thật Hibernate sinh ra, với tham số thật, đúng yêu cầu plan 3.1.3.
3. **`no_filter`:** gọi **một lần sau cùng**, vừa ghi plan vừa kiểm tra kết quả; bản có phân trang gọi thêm `no_filter_page0` sau đó.

Cột `generic_plan` cho biết plan có dùng tham số dạng `$1` hay không. pgJDBC chuyển sang prepared statement phía server sau vài lần chạy. Từ lần thứ sáu của prepared statement, PostgreSQL chỉ chọn generic plan (plan chung cho mọi giá trị tham số) khi cost của nó thấp hơn **trung bình cost các custom plan trước đó** trên cùng connection.

Vì vậy `no_filter` phải chạy sau cùng. Plan của nó có cost cao gấp nhiều lần các bộ lọc khác; chạy sớm, nó kéo trung bình lên và connection chuyển sang generic plan cho mọi request sau. Với truy vấn dạng `(:skip = true OR …)`, generic plan không dùng được index theo ngày. Lượt đo tải không gọi `no_filter`, nên thứ tự này giữ plan ghi được giống plan lúc đo. Lần chạy P1 `phase2-a-20261118T175519Z` còn gọi `no_filter` trước các vòng trace, nên trace của lần đó gần như chỉ có generic plan.

In [ ]:
def walk(node):
    yield node
    for child in node.get('Plans', []):
        yield from walk(child)


def plan_summary(entry):
    plan = entry['plan']['Plan']
    scans = [f"{n['Node Type']} {n.get('Relation Name', '')} {n.get('Index Name', '')}".strip()
             for n in walk(plan) if 'Relation Name' in n or 'Index Name' in n]
    return {'duration_ms': entry['duration_ms'], 'generic_plan': '$1' in json.dumps(plan),
            'rows': plan.get('Actual Rows'), 'shared_hit_blocks': plan.get('Shared Hit Blocks'),
            'shared_read_blocks': plan.get('Shared Read Blocks'), 'scans': ' | '.join(scans),
            'query': entry['plan'].get('Query Text', '')[:4000],
            'parameters': entry['plan'].get('Query Parameters', '')}


def read_plans(since):
    entries = []
    for path in sorted(PG_LOG_DIR.glob('*.json')):
        for line in path.read_text(encoding='utf-8', errors='replace').splitlines():
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                continue
            message = record.get('message', '')
            if 'plan:' not in message:
                continue
            stamp = datetime.strptime(record['timestamp'].replace(' UTC', ''), '%Y-%m-%d %H:%M:%S.%f').replace(tzinfo=timezone.utc)
            if stamp < since:
                continue
            duration = re.search(r'duration: ([0-9.]+) ms', message)
            try:
                plan = json.loads(message.split('plan:', 1)[1])
            except json.JSONDecodeError:
                continue
            entries.append({'timestamp': stamp, 'duration_ms': float(duration[1]) if duration else None, 'plan': plan})
    return entries


def check_response(variant, name, body):
    params, expected = FILTERS[name]
    if len(body) != expected:
        raise RuntimeError(f'{variant}/{name}: trả {len(body)} chuyến, SQL đếm được {expected}. '
                           'Nếu lệch ở bộ lọc ngày, kiểm tra múi giờ khi so sánh departure_time.')
    times = [trip['departureTime'] for trip in body]
    if variant in PAGED_VARIANTS:
        keys = [(trip['departureTime'], trip['id']) for trip in body]
        if any(a >= b for a, b in zip(keys, keys[1:])):
            raise RuntimeError(f'{variant}/{name}: chuyến không theo thứ tự departureTime rồi id')
    elif times != sorted(times):
        raise RuntimeError(f'{variant}/{name}: departureTime không theo thứ tự tăng dần')
    if 'departureDate' in params and any(not t.startswith(params['departureDate']) for t in times):
        raise RuntimeError(f'{variant}/{name}: có chuyến không thuộc ngày {params["departureDate"]}')
    if name in FILTER_ROUTE and any(trip['routeId'] != FILTER_ROUTE[name] for trip in body):
        raise RuntimeError(f'{variant}/{name}: có chuyến không thuộc tuyến {FILTER_ROUTE[name]}')
    ids = [trip['id'] for trip in body]
    if name in FIRST_PAGE_IDS and ids != FIRST_PAGE_IDS[name]:
        raise RuntimeError(f'{variant}/{name}: trang đầu khác {PAGE_SIZE} ID đầu tiên theo departure_time, id')
    return ids


def digest(trips):
    return hashlib.sha256(json.dumps(trips, sort_keys=True, ensure_ascii=False).encode('utf-8')).hexdigest()


RESPONSE_IDS = {}
RESPONSE_DIGESTS = {}
TRACE_SUMMARY = []
for variant, _ in VARIANTS:
    print(f'Kiểm tra kết quả và ghi query plan: {variant}', flush=True)
    out = OUTPUT / 'trace' / variant
    db = f'busb_a_trace_{variant.lower()}'
    with connect() as admin:
        admin.execute(sql.SQL('CREATE DATABASE {} TEMPLATE {} OWNER {}').format(
            sql.Identifier(db), sql.Identifier(BASE_DB), sql.Identifier(APP_DB_USER)))
        for setting, value in (('auto_explain.log_min_duration', '0'), ('auto_explain.log_analyze', 'on'),
                               ('auto_explain.log_buffers', 'on'), ('auto_explain.log_format', 'json')):
            admin.execute(sql.SQL('ALTER DATABASE {} SET {} = {}').format(
                sql.Identifier(db), sql.Identifier(*setting.split('.')), sql.Literal(value)))
    try:
        with backend(variant, db, out / 'backend.log'):
            with connect(db) as connection:
                connection.execute('ANALYZE')
                write_json(out / 'flyway.json', flyway_versions(connection))
            # Cùng thứ tự với lượt đo tải: no_filter (cost rất cao) chạy sau cùng, để không kéo
            # trung bình cost custom plan lên và đẩy connection sang generic plan.
            RESPONSE_IDS[variant] = {}
            RESPONSE_DIGESTS[variant] = {}
            windows = []
            filters = load_filters(variant)
            last = ['no_filter'] + (['no_filter_page0'] if variant in PAGED_VARIANTS else [])
            sequence = ([(name, None) for name in filters]
                        + [(name, round_index + 1) for round_index in range(TRACE_ROUNDS) for name in filters]
                        + [(name, 1) for name in last])
            for name, round_index in sequence:
                start = datetime.now(timezone.utc)
                response = requests.get(API + filter_path(name), timeout=120)
                response.raise_for_status()
                if round_index is not None:
                    windows.append((name, round_index, start, datetime.now(timezone.utc)))
                if name not in RESPONSE_IDS[variant]:
                    body = response.json()
                    RESPONSE_IDS[variant][name] = check_response(variant, name, body)
                    RESPONSE_DIGESTS[variant][name] = digest(sorted(body, key=lambda trip: trip['id']))
                    if variant == 'P1':
                        # Trang đầu mong đợi: kết quả đầy đủ của P1 sắp theo departureTime rồi id.
                        for page_name, base in PAGE_OF.items():
                            if base == name:
                                page = sorted(body, key=lambda trip: (trip['departureTime'], trip['id']))[:PAGE_SIZE]
                                RESPONSE_DIGESTS[variant][page_name] = digest(sorted(page, key=lambda trip: trip['id']))
                    print(f'  {name}: {len(response.content):,} byte, {len(RESPONSE_IDS[variant][name])} chuyến')
            write_json(out / 'response_ids.json', RESPONSE_IDS[variant])
            write_json(out / 'response_digests.json', RESPONSE_DIGESTS[variant])
            time.sleep(2)
        entries = read_plans(windows[0][2])
        plan_rows = []
        with (out / 'plans.jsonl').open('w', encoding='utf-8') as raw:
            for name, round_index, start, end in windows:
                for entry in entries:
                    if start <= entry['timestamp'] <= end:
                        raw.write(json.dumps({'filter': name, 'round': round_index, **entry}, default=str) + '\n')
                        plan_rows.append({'variant': variant, 'filter': name, 'round': round_index, **plan_summary(entry)})
        write_csv(out / 'plans.csv', plan_rows)
        if not plan_rows:
            print('  CẢNH BÁO: không đọc được plan nào từ log PostgreSQL; xem thư mục log trong ZIP.')
        for name in filters + last:
            rows = [row for row in plan_rows if row['filter'] == name]
            if rows:
                TRACE_SUMMARY.append({
                    'variant': variant, 'filter': name, 'statements': len(rows),
                    'generic_plans': sum(row['generic_plan'] for row in rows),
                    'median_sql_ms': median(row['duration_ms'] for row in rows if row['duration_ms'] is not None),
                    'rows_last': rows[-1]['rows'], 'shared_hit_last': rows[-1]['shared_hit_blocks'],
                    'shared_read_last': rows[-1]['shared_read_blocks'], 'scans_first': rows[0]['scans'],
                    'scans_last': rows[-1]['scans']})
    finally:
        with connect() as admin:
            admin.execute(sql.SQL('DROP DATABASE IF EXISTS {} WITH (FORCE)').format(sql.Identifier(db)))

for variant, _ in VARIANTS[1:]:
    for name in RESPONSE_IDS['P1']:
        if sorted(RESPONSE_IDS[variant][name]) != sorted(RESPONSE_IDS['P1'][name]):
            raise RuntimeError(f'{variant}/{name} trả tập chuyến khác P1')
    for name, value in RESPONSE_DIGESTS[variant].items():
        if value != RESPONSE_DIGESTS['P1'][name]:
            raise RuntimeError(f'{variant}/{name}: JSON của chuyến khác P1')
write_csv(OUTPUT / 'trace_summary.csv', TRACE_SUMMARY)
from IPython.display import Markdown, display
if TRACE_SUMMARY:
    keys = list(TRACE_SUMMARY[0])
    display(Markdown('\n'.join(['| ' + ' | '.join(keys) + ' |', '|' + '---|' * len(keys)] +
                               ['| ' + ' | '.join(str(row[key]) for key in keys) + ' |' for row in TRACE_SUMMARY])))

## 8. Đo tải GET tìm chuyến

Mỗi lượt, mỗi bản:
1. Tạo database mới từ snapshot, khởi động backend (Flyway của bản đó chạy tại đây), `ANALYZE`.
2. Warm-up `WARMUP_SECONDS` giây ở mức users cao nhất, trộn mọi bộ lọc đo tải; kết quả warm-up bị loại.
3. Với từng bộ lọc (bản có phân trang thêm `large_gt_20_page0`, `date_only_page0`) và từng mức users: chụp `pg_stat_statements`, chạy Locust `MEASURE_SECONDS` giây, chụp lại để lấy chênh lệch SQL; đồng thời lấy mẫu CPU/RSS.
4. Dừng backend, xóa database. Lượt lẻ chạy theo thứ tự `VARIANTS`, lượt chẵn đảo ngược.

**Giới hạn:** Locust, Java và PostgreSQL dùng chung CPU của Kaggle. Cột `locust_cpu_cores_avg` gần 1,0 nghĩa là Locust có thể đang là điểm nghẽn (Locust chạy một tiến trình).

In [ ]:
load_runs = REPETITIONS * sum(len(load_filters(variant)) for variant, _ in VARIANTS) * len(USER_LEVELS)
estimate = REPETITIONS * len(VARIANTS) * (40 + WARMUP_SECONDS) + load_runs * (MEASURE_SECONDS + 5)
print(f'{load_runs} lượt đo, ước tính khoảng {estimate / 60:.0f} phút', flush=True)

SUMMARY = []
for repetition in range(1, REPETITIONS + 1):
    order = VARIANTS if repetition % 2 else list(reversed(VARIANTS))
    for variant, _ in order:
        out = OUTPUT / 'runs' / f'{repetition:02d}_{variant}'
        db = f'busb_a_run_{variant.lower()}'
        with connect() as admin:
            admin.execute(sql.SQL('CREATE DATABASE {} TEMPLATE {} OWNER {}').format(
                sql.Identifier(db), sql.Identifier(BASE_DB), sql.Identifier(APP_DB_USER)))
        try:
            with backend(variant, db, out / 'backend.log') as process:
                with connect(db) as connection:
                    connection.execute('ANALYZE')
                    versions = flyway_versions(connection)
                    write_json(out / 'flyway.json', versions)
                    paths = [[name, filter_path(name)] for name in load_filters(variant)]
                    print(f'Lượt {repetition}/{REPETITIONS} | {variant} | warm-up {WARMUP_SECONDS}s', flush=True)
                    warm = run_locust(paths, max(USER_LEVELS), WARMUP_SECONDS, out / 'warmup', process.pid, False)
                    if warm['failures']:
                        raise RuntimeError(f'Warm-up có {warm["failures"]} request lỗi, xem {out}')
                    for name in load_filters(variant):
                        for users in USER_LEVELS:
                            label = f'{name}_u{users}'
                            print(f'Lượt {repetition}/{REPETITIONS} | {variant} | {label}', flush=True)
                            before = statements(connection)
                            started_at = utc()
                            result = run_locust([[name, filter_path(name)]], users, MEASURE_SECONDS, out / label,
                                                process.pid, True)
                            sql_rows = statement_delta(before, statements(connection))
                            write_csv(out / f'{label}_sql.csv', sql_rows,
                                      ['queryid', 'calls', 'exec_ms', 'rows', 'shared_blks_hit', 'shared_blks_read', 'query'])
                            ok = result['requests'] - result['failures']
                            row = {'repetition': repetition, 'variant': variant, 'filter': name, 'users': users,
                                   'expected_rows': FILTERS[name][1], 'started_at_utc': started_at,
                                   'flyway_latest': versions[-1] if versions else None, **result,
                                   'sql_calls': sum(r['calls'] for r in sql_rows),
                                   'sql_exec_ms': sum(r['exec_ms'] for r in sql_rows),
                                   'sql_rows': sum(r['rows'] for r in sql_rows),
                                   'shared_blks_hit': sum(r['shared_blks_hit'] for r in sql_rows),
                                   'shared_blks_read': sum(r['shared_blks_read'] for r in sql_rows)}
                            for key in ('sql_calls', 'sql_exec_ms', 'sql_rows', 'shared_blks_hit', 'shared_blks_read'):
                                row[f'{key}_per_request'] = row[key] / ok if ok else None
                            SUMMARY.append(row)
                            write_csv(OUTPUT / 'summary.csv', SUMMARY)
                            print(f"   p50={row['p50_ms']} ms  p95={row['p95_ms']} ms  rps={row['rps']:.1f}  "
                                  f"lỗi={row['failures']}  SQL/request={row['sql_calls_per_request']}", flush=True)
        finally:
            with connect() as admin:
                admin.execute(sql.SQL('DROP DATABASE IF EXISTS {} WITH (FORCE)').format(sql.Identifier(db)))
print('Xong đo tải.')

## 9. Tổng hợp, manifest và đóng gói ZIP

`aggregate.csv` ghi **trung vị và min–max qua các lượt** cho từng bản × bộ lọc × mức users. Trung vị của p95 qua các lượt khác với p95 gộp mọi request. Chỉ nhận cải thiện khi khoảng min–max của hai bản không chồng nhau và xu hướng lặp lại (plan 6.3).

In [ ]:
import importlib.metadata

AGG_METRICS = ['p50_ms', 'p95_ms', 'p99_ms', 'successful_rps', 'avg_response_bytes', 'sql_calls_per_request',
               'sql_exec_ms_per_request', 'sql_rows_per_request', 'shared_blks_hit_per_request',
               'java_cpu_cores_avg', 'postgresql_cpu_cores_avg', 'locust_cpu_cores_avg', 'java_peak_rss_mib']
AGGREGATE = []
for variant, _ in VARIANTS:
    for name in load_filters(variant):
        for users in USER_LEVELS:
            rows = [r for r in SUMMARY if (r['variant'], r['filter'], r['users']) == (variant, name, users)]
            for metric in AGG_METRICS:
                values = [r[metric] for r in rows if r.get(metric) is not None]
                if values:
                    AGGREGATE.append({'variant': variant, 'filter': name, 'users': users, 'metric': metric,
                                      'trials': len(values), 'median': median(values),
                                      'min': min(values), 'max': max(values)})
write_csv(OUTPUT / 'aggregate.csv', AGGREGATE)

cpuinfo = Path('/proc/cpuinfo')
model = re.search(r'^model name\s*:\s*(.+)$', cpuinfo.read_text(), re.MULTILINE) if cpuinfo.is_file() else None
MANIFEST = {
    'run_id': RUN_ID, 'end_utc': utc(), 'quick_check': QUICK_CHECK,
    'purpose': 'Phase 2A trip search read path: P1 baseline and improvement variants in one Kaggle session',
    'platform': platform.platform(), 'cpu_model': model[1] if model else platform.processor(),
    'cpu_logical': psutil.cpu_count(), 'ram_bytes': psutil.virtual_memory().total,
    'jdk': JAVA_VERSION, 'java_flags': JAVA_FLAGS, 'hikari_max_pool': HIKARI_POOL,
    'python': platform.python_version(),
    'packages': {name: importlib.metadata.version(name) for name in ('locust', 'psutil', 'psycopg', 'requests')},
    'postgresql': PG_SETTINGS, 'repository': REPO_URL, 'variants': SOURCES,
    'settings': {'user_levels': USER_LEVELS, 'wait_seconds': 0, 'spawn_rate': 'equal to users',
                 'warmup_seconds': WARMUP_SECONDS, 'measure_seconds': MEASURE_SECONDS,
                 'repetitions': REPETITIONS, 'trace_rounds': TRACE_ROUNDS, 'order': 'VARIANTS order on odd runs, reversed on even runs',
                 'load_filters': LOAD_FILTERS, 'paged_variants': sorted(PAGED_VARIANTS & set(SOURCES)),
                 'page_size': PAGE_SIZE, 'paged_load_filters': PAGED_LOAD_FILTERS},
    'fixture': FIXTURE_STATS,
    'input_sha256': {'locustfile.py': sha256_file(LOCUSTFILE), 'fixture.sql': sha256_file(OUTPUT / 'fixture.sql')},
    'measurement_limitations': ['Locust, Java and PostgreSQL share the Kaggle CPU',
                                'CPU model can differ across Kaggle sessions; compare variants only within one run',
                                'Plans come from a separate traced database with auto_explain enabled, not from the load runs'],
}
write_json(OUTPUT / 'manifest.json', MANIFEST)

pg_log_out = OUTPUT / 'postgresql_log'
pg_log_out.mkdir(exist_ok=True)
for path in PG_LOG_DIR.glob('*'):
    shutil.copyfile(path, pg_log_out / path.name)
ZIP_PATH = shutil.make_archive(str(OUTPUT), 'zip', root_dir=OUTPUT.parent, base_dir=OUTPUT.name)

from IPython.display import FileLink, Markdown, display
table = ['| Bản | Bộ lọc | Users | p95 ms (min–max) | RPS | Byte/response | SQL ms/request |', '|---|---|---:|---:|---:|---:|---:|']
for variant, _ in VARIANTS:
    for name in load_filters(variant):
        for users in USER_LEVELS:
            cell = {a['metric']: a for a in AGGREGATE if (a['variant'], a['filter'], a['users']) == (variant, name, users)}
            if 'p95_ms' not in cell:
                continue
            p95, rps = cell['p95_ms'], cell['successful_rps']
            size, sql_ms = cell.get('avg_response_bytes'), cell.get('sql_exec_ms_per_request')
            table.append(f"| {variant} | {name} | {users} | {p95['median']:.1f} ({p95['min']:.1f}–{p95['max']:.1f}) | "
                         f"{rps['median']:.1f} | {size['median'] if size else float('nan'):,.0f} | "
                         f"{sql_ms['median'] if sql_ms else float('nan'):.3f} |")
display(Markdown('\n'.join(table)))
print('ZIP:', ZIP_PATH)
display(FileLink(os.path.relpath(ZIP_PATH, WORKING)))

## 10. Dừng PostgreSQL

Chạy cell này khi xong. Nếu một cell ở trên báo lỗi giữa chừng, đọc lỗi, sửa cấu hình rồi **Restart & Run All** để bắt đầu lại từ trạng thái sạch.

In [ ]:
run([*AS_POSTGRES, PG_BIN / 'pg_ctl', '-D', PG_DATA, '-m', 'fast', '-w', 'stop'])
print('Đã dừng PostgreSQL.')